
# 🧠 Highlight → Topic → Cluster Pipeline (v4)



## Overview
* **Composite embedding** = CLS(span) ‖ CLS(paragraph) ‖ CLS(chapter)  
* **Topics** = deduplicated highlights (cos sim ≥ `dup_thr`)  
* **Clusters** = HDBSCAN over Topic centroids  
* Vectors are **purely semantic**; evidence (`pos/neg`) is tracked separately  
* Summary prints *every* cluster, its topics, and the raw highlight texts.



## 0&nbsp;·&nbsp;Install (run once)

```bash
!pip install transformers torch hdbscan scikit-learn networkx matplotlib tqdm
```


In [1]:
## 📦 Imports & Dependencies
# This cell loads all Python standard libraries and ML frameworks required throughout the notebook, including Transformers for BERT embeddings and NetworkX for graph utilities.

import json, collections, math, re
from pathlib import Path
import numpy as np, hdbscan, torch, networkx as nx, matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from transformers import BertTokenizer, BertModel


/Users/gon/Documents/Spring 2025/H2AI/v2/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## 🛠️ Text‑processing Helpers
# Utility functions to split raw documents into chapters & paragraphs and locate a highlight’s surrounding context.

def split_into_chapters(text, max_words=400):
    words, cur, out = text.split(), [], []
    for w in words:
        cur.append(w)
        if len(cur) >= max_words:
            out.append(" ".join(cur)); cur=[]
    if cur: out.append(" ".join(cur))
    return out

def paragraphs(ch_text):
    ps=[p.strip() for p in ch_text.split("\n\n") if p.strip()]
    return ps if ps else [ch_text]

def locate_context(span, chapters):
    for ci, ch in enumerate(chapters):
        if span in ch:
            for p in paragraphs(ch):
                if span in p:
                    return ci, ch, p
    return 0, chapters[0], paragraphs(chapters[0])[0]


In [3]:
## 🔍 BertEncoder Class
# Lightweight wrapper around Hugging Face BERT to generate a single 768‑dimensional embedding for any text span.

class BertEncoder:
    def __init__(self, model='bert-base-uncased', device=None):
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.tok = BertTokenizer.from_pretrained(model)
        self.net = BertModel.from_pretrained(model).to(self.device)
        self.net.eval()
        for p in self.net.parameters(): p.requires_grad=False

    @torch.inference_mode()
    def emb(self, texts):
        toks=self.tok(texts, padding=True, truncation=True, max_length=128,
                      return_tensors='pt').to(self.device)
        out=self.net(**toks)
        return out.last_hidden_state[:,0,:].cpu().numpy()


In [4]:
## 🗂️ Topic Data Structure
# Maintains statistics for one semantic ‘topic’: running centroid vector, highlight strings, chapter references, and belief (understood vs not_understood).

class Topic:
    def __init__(self, vec, span, chap, understood):
        self.vec=vec.copy()
        self.highlights=[span]
        self.chaps={chap}
        self.pos=int(understood); self.neg=int(not understood)
    @property
    def belief(self): return self.pos/(self.pos+self.neg)

    def add(self, vec, span, chap, understood):
        k=len(self.highlights)+1
        self.vec=(self.vec*(k-1)+vec)/k
        self.highlights.append(span)
        self.chaps.add(chap)
        if understood: self.pos+=1
        else: self.neg+=1


In [5]:
## 🧠 KB (Knowledge Base)
# Deduplicates topics using cosine similarity, then groups them into higher‑level clusters with agglomerative clustering.

class KB:
    def __init__(self, dup_thr=0.9):
        self.dup_thr=dup_thr; self.topics=[]; self.cluster_labels=[]
    def add(self, vec, span, chap, understood):
        if self.topics:
            sims=cosine_similarity([vec],[t.vec for t in self.topics])[0]
            j=int(sims.argmax())
            if sims[j]>=self.dup_thr:
                self.topics[j].add(vec, span, chap, understood); return
        self.topics.append(Topic(vec, span, chap, understood))
    def recluster(self, min_samples=1):
        if len(self.topics) < 2:
            self.cluster_labels = [-1] * len(self.topics)
            return
            
        # Option 1: Use cosine distance directly (works better with semantic vectors)
        embs = np.stack([t.vec for t in self.topics])
        
        # Try one of these approaches:
        
        # A. For very small datasets, simple AgglomerativeClustering
        from sklearn.cluster import AgglomerativeClustering
        model = AgglomerativeClustering(
            n_clusters=None,
            distance_threshold=0.5,  # Adjust based on your similarity needs
            metric='cosine',
            linkage='average'
        )
        self.cluster_labels = model.fit_predict(embs)
    def clusters(self):
        d=collections.defaultdict(list)
        for i,l in enumerate(self.cluster_labels): d[l].append(i)
        return d


In [6]:
## 🚀 Pipeline Orchestrator
# Connects encoder + KB into a single `.ingest()` method for new documents and a `.summary()` reporter.

class Pipeline:
    def __init__(self, dup_thr=0.9):
        self.enc=BertEncoder()
        self.kb=KB(dup_thr)
    def ingest(self, doc):
        chaps=split_into_chapters(doc['text'])
        for hl in doc['highlights']:
            span=hl['span']; understood=hl['category']=='understood'
            ci,ch_text,par=locate_context(span, chaps)
            v_span,v_par,v_chap=self.enc.emb([span,par,ch_text])
            vec=np.concatenate([v_span,v_par,v_chap],0)
            self.kb.add(vec, span, ci, understood)
        self.kb.recluster()
    def summary(self, max_hl=4):
        # Count actual clusters (excluding noise points with label -1)
        valid_clusters = [label for label in self.kb.cluster_labels if label != -1]
        num_clusters = len(set(valid_clusters)) if valid_clusters else 0
        
        out=[f"Topics: {len(self.kb.topics)} | Clusters: {num_clusters}"]
        
        for cid,idxs in self.kb.clusters().items():
            if cid==-1:
                # Optionally add a section for unclustered topics
                out.append(f"\nUnclustered Topics ({len(idxs)})")
                for i in idxs[:2]:  # Show just a few examples
                    t=self.kb.topics[i]
                    out.append(f"  • Topic {i} | belief {t.belief:.2f} | chaps {sorted(t.chaps)}")
                    for s in t.highlights[:1]:
                        out.append(f"     - {s[:120]}")
                continue
                
            pos=sum(self.kb.topics[i].pos for i in idxs)
            neg=sum(self.kb.topics[i].neg for i in idxs)
            belief=pos/(pos+neg)
            out.append(f"\nCluster {cid}  pos {pos} neg {neg} belief {belief:.2f}")
            
            for i in idxs:
                t=self.kb.topics[i]
                out.append(f"  • Topic {i} | belief {t.belief:.2f} | chaps {sorted(t.chaps)}")
                for s in t.highlights[:max_hl]:
                    out.append(f"     - {s[:120]}")
        return "\n".join(out)


In [7]:
## 🎯 Quick Demo
# Runs the pipeline on a few synthetic JSON docs and prints the resulting topic / cluster summary.

# Demo with synthetic docs (ensure paths exist)
doc_paths=[Path('data/doc4.json'),
           Path('data/doc5.json'),
           Path('data/doc6.json')]
pipe=Pipeline()
for p in doc_paths:
    print(f'\n=== {p.name} ===')
    pipe.ingest(json.loads(p.read_text()))
    print(pipe.summary())



=== doc4.json ===
Topics: 1 | Clusters: 0

Unclustered Topics (1)
  • Topic 0 | belief 0.50 | chaps [0]
     - Convolutional Neural Networks, or CNNs, have revolutionized computer vision by exploiting local pixel patterns.

=== doc5.json ===
Topics: 2 | Clusters: 1

Cluster 0  pos 10 neg 10 belief 0.50
  • Topic 0 | belief 0.50 | chaps [0]
     - Convolutional Neural Networks, or CNNs, have revolutionized computer vision by exploiting local pixel patterns.
     - The receptive field of a convolutional neuron expands with network depth and kernel size.
     - Pooling layers reduce spatial resolution while promoting translational invariance.
     - Dilated convolutions enlarge receptive fields without increasing parameter count.
  • Topic 1 | belief 0.50 | chaps [0]
     - Reinforcement Learning formalizes sequential decision making through agent environment interaction.
     - The objective is to maximize expected discounted return over time.
     - Q-learning estimates the value of st

In [ ]:
## (Code)
